In [0]:
# ============================================================
# Bronze — Source 14: Scrapy (Competitor Pricing)
#
# Incremental load using watermark pattern.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=14_scrapy/
# Target: bronze.scrapy catalog in Unity Catalog
#
# Tables:
#   - bronze.scrapy.competitor_pricing
#
# Raw format: JSON (daily partitioned)
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "14_scrapy"

dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=18/")

In [0]:
# Cell 2 — Read competitor pricing JSON files
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/competitor_prices_*.json"

df = spark.read.format("json") \
    .load(path) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

print(f"Raw rows: {df.count()}")
df.printSchema()

In [0]:
# Cell 3 — Flatten records and write to Bronze
from pyspark.sql.functions import explode

pricing_flat = df \
    .select(explode(col("records")).alias("r")) \
    .select(
        col("r.product_sku"),
        col("r.product_title"),
        col("r.our_price_pence"),
        col("r.competitor_id"),
        col("r.competitor_name"),
        col("r.competitor_domain"),
        col("r.competitor_sku"),
        col("r.price_pence"),
        col("r.price_difference_pct"),
        col("r.currency"),
        col("r.availability"),
        col("r.promo_active"),
        col("r._source").alias("source"),
    )

print(f"Flattened rows: {pricing_flat.count()}")

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.scrapy")

pricing_flat.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.scrapy.competitor_pricing")

latest_ts = df.select(spark_max("_metadata.file_modification_time")).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, pricing_flat.count())
print(f"✅ bronze.scrapy.competitor_pricing: {pricing_flat.count()} rows")

In [0]:
count = spark.sql("SELECT COUNT(*) as cnt FROM bronze.scrapy.competitor_pricing").collect()[0]['cnt']
print(f"bronze.scrapy.competitor_pricing: {count} rows")
spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '14_scrapy'").show()